# 03 — Spatial fold maps over a synthetic grid

This notebook projects each fold's `SplitRegions` onto a synthetic two-dimensional azimuth-by-range grid and paints every pixel with its split role. This is the spatial view of the assignment: it confirms that the `CropRegion` azimuth bounds produced by `FoldPlanner.plan` tile the grid without leakage along the azimuth axis, and that range is never split (every fold spans the full range extent).

Modules exercised: `pipelines.cross_validation_pipeline.folds.FoldPlanner`, `tools.regions.SplitRegions`, `tools.regions.CropRegion`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## Planner over a small grid and a painting helper

In [ ]:
from configuration.cross_validation_config import CrossValidationConfig
from pipelines.cross_validation_pipeline.folds import FoldPlanner

AZ_START, AZ_END = 0, 240
RG_START, RG_END = 0, 60

config                     = CrossValidationConfig()
config.folds.n_folds       = 6
config.folds.azimuth_start = AZ_START
config.folds.azimuth_end   = AZ_END

planner = FoldPlanner(config, range_start=RG_START, range_end=RG_END)

ROLE = {"train": 0, "val": 1, "test": 2}

def paint(plan):
    grid = np.full((RG_END - RG_START, AZ_END - AZ_START), ROLE["train"], dtype=int)
    for split in ("val", "test"):
        for region in plan.split_regions.regions(split):
            az = slice(region.azimuth_start - AZ_START, region.azimuth_end - AZ_START)
            rg = slice(region.range_start   - RG_START, region.range_end   - RG_START)
            grid[rg, az] = ROLE[split]
    return grid

## Paint and display the fold maps

One panel per fold. The horizontal axis is azimuth, the vertical axis is range. The test band sweeps across the azimuth axis as the fold index advances, with the validation band immediately following it.

In [ ]:
import matplotlib.colors as mcolors

cmap  = mcolors.ListedColormap(["#cfe8cf", "#f4c879", "#d96c6c"])
plans = planner.plans()

fig, axes = plt.subplots(len(plans), 1, figsize=(9, 1.5 * len(plans)), sharex=True)

for ax, plan in zip(axes, plans):
    ax.imshow(paint(plan), cmap=cmap, aspect="auto", vmin=0, vmax=2, origin="lower", extent=[AZ_START, AZ_END, RG_START, RG_END])
    ax.set_ylabel(f"fold {plan.fold_index}\nrange")

axes[-1].set_xlabel("azimuth (samples)")
axes[0].set_title("Spatial fold maps (green train, amber val, red test)")
plt.tight_layout()
plt.show()

## Numerical leakage check on the painted grids

We recount the pixels of each role directly from the painted grids and confirm they sum to the full grid area, and that the test plus validation pixels each match one block width times the full range height.

In [ ]:
grid_area  = (AZ_END - AZ_START) * (RG_END - RG_START)
block_area = (planner.blocks[0][1] - planner.blocks[0][0]) * (RG_END - RG_START)

for plan in plans:
    grid    = paint(plan)
    n_train = int((grid == ROLE["train"]).sum())
    n_val   = int((grid == ROLE["val"]).sum())
    n_test  = int((grid == ROLE["test"]).sum())
    print(f"fold {plan.fold_index}: train={n_train:5d} val={n_val:5d} test={n_test:5d} sum_ok={n_train + n_val + n_test == grid_area}")

## Expected visual outcome

Six stacked maps in which the red test band marches left to right one block per fold, trailed by the amber validation band, over a green training background. The range axis is never subdivided. The pixel counts sum exactly to the grid area for every fold, confirming no leakage and full coverage.